# Solution E2 · Coefficients with the wrong names: the column order after the ColumnTransformer

**Type**: Find the bug

## Objective

Discover that a variable importance analysis gives a wrong answer because the names of the original columns were
matched to the coefficients in a different order from the one in which the `ColumnTransformer` produced them, and rebuild the right names.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]]
y = df["defaulted"]

ct = ColumnTransformer([("a", StandardScaler(), ["income", "amount"]),
                        ("b", StandardScaler(), ["age", "job_tenure"])])
Z = ct.fit_transform(X)
model = LogisticRegression(max_iter=2000).fit(Z, y)

## Commented solution

In [ ]:
# the colleague's code: it runs, but it is wrong
coef_bug = pd.Series(model.coef_[0], index=X.columns)          # X.columns = age, income, job_tenure, amount
bug_variable = coef_bug.abs().idxmax()
print(coef_bug.round(2).to_string())
print("most influential variable according to the colleague:", bug_variable)

**Where the bug is.** `Z` has the columns in the order of the transformers (`income, amount, age, job_tenure`), but the
colleague used `X.columns` (`age, income, job_tenure, amount`) as the index: the coefficient of income was labeled
`age`, that of amount `income`, and so on. No error: it is four numbers and four names.

In [ ]:
correct_names = [n.split("__")[-1] for n in ct.get_feature_names_out()]
coef_ok = pd.Series(model.coef_[0], index=correct_names)
right_variable = coef_ok.abs().idxmax()
income_sign = "positive" if coef_ok["income"] > 0 else "negative"
amount_sign = "positive" if coef_ok["amount"] > 0 else "negative"
print(pd.DataFrame({"wrong": coef_bug.values, "correct": coef_ok.reindex(X.columns).values}, index=X.columns).round(2))
print("most influential: colleague", bug_variable, "| correct", right_variable)
print("income sign:", income_sign, "| amount sign:", amount_sign)

## Check

In [ ]:
with verify():
    check(correct_names == ["income", "amount", "age", "job_tenure"], "names rebuilt in the output order of the ColumnTransformer",
          "the order is: income, amount, age, job_tenure (get_feature_names_out, without prefix)")
    check(len(coef_ok) == 4 and list(coef_ok.index) == correct_names, "coef_ok has the correct names as index", "coef_ok = pd.Series(model.coef_[0], index=correct_names)")
    check(close(coef_ok.to_numpy(), model.coef_[0]), "the coefficients are the same: only the names change", "don't change the values, only the matching to the names")
    check(right_variable == coef_ok.abs().idxmax() and right_variable != bug_variable,
          "the most influential variable is a different one from the one declared by the colleague", "right_variable = coef_ok.abs().idxmax()")
    check(income_sign == "negative" and amount_sign == "positive",
          "a higher income lowers the risk, a higher amount raises it: sensible signs",
          "look at the sign of the coefficients of income and amount in coef_ok")
done()

## What you learned

- A transformer returns an array **without names** and in the order of its blocks, not of the original columns. Matching the
  original names to the coefficients "by position" is a silent mistake: four numbers, four names, no warning.
- `get_feature_names_out()` is the right way to rebuild the names (with the step prefixes: `a__income`, `b__age`...). In a
  `Pipeline` you get it with `pipe[:-1].get_feature_names_out()`.
- A plausibility test helps: a higher income must lower the risk (negative sign), a higher amount must raise it.
  If the signs are counterintuitive, the first hypothesis is a labeling error, before a discovery.

## Common pitfall

Trusting the order of the columns "because it is the same as before anyway". With `remainder="passthrough"`, with transformers that
change the number of columns (one-hot) or with blocks in a different order, that assumption is false. The rule: names are
rebuilt from the transformer, never by hand.

## Interview question

<details><summary>How do you get the feature names after a Pipeline with ColumnTransformer and OneHotEncoder?</summary>

With `get_feature_names_out()` on the preprocessor (`pipe[:-1].get_feature_names_out()` or `pipe.named_steps['prep']...`), which returns the names
in the order of the output columns, with the step prefix (`num__income`, `cat__city_Rome`). I use them to label
coefficients or importances; I never rebuild them by hand, because the order depends on the blocks and on the number of categories.

</details>